Puedes comprar versiones impresas y libro electrónico de *Think Python 3e* en
[Bookshop.org](https://bookshop.org/a/98697/9781098155438) y
[Amazon](https://www.amazon.com/_/dp/1098155432?smid=ATVPDKIKX0DER&_encoding=UTF8&tag=oreilly20-20&_encoding=UTF8&tag=greenteapre01-20&linkCode=ur2&linkId=e2a529f94920295d27ec8a06e757dc7c&camp=1789&creative=9325).

In [1]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');

import thinkpython

# Cadenas y expresiones regulares

Las cadenas no son como los enteros, los flotantes y los booleanos. Una cadena es una **secuencia**, lo que significa que contiene múltiples valores en un orden particular.
En este capítulo veremos cómo acceder a los valores que componen una cadena, y usaremos funciones que procesan cadenas.

También usaremos expresiones regulares, que son una herramienta poderosa para encontrar patrones en una cadena y realizar operaciones como búsqueda y reemplazo.

Como ejercicio, tendrás la oportunidad de aplicar estas herramientas a un juego de palabras llamado Wordle.

## Una cadena es una secuencia

Una cadena es una secuencia de caracteres. Un **carácter** puede ser una letra (en casi cualquier alfabeto), un dígito, un signo de puntuación o un espacio en blanco.

Puedes seleccionar un carácter de una cadena con el operador de corchetes.
Esta sentencia de ejemplo selecciona el carácter número 1 de `fruit` y
lo asigna a `letter`:

In [2]:
fruit = 'banana'
letter = fruit[1]

La expresión entre corchetes es un **índice**, llamado así porque *indica* qué carácter de la secuencia seleccionar.
Pero el resultado quizá no sea lo que esperas.

In [3]:
letter

La letra con índice `1` es en realidad la segunda letra de la cadena.
Un índice es un desplazamiento desde el principio de la cadena, así que el desplazamiento de la primera letra es `0`.

In [4]:
fruit[0]

Puedes pensar en `'b'` como la letra 0 de `'banana'` -- pronunciada "ceroésima".

El índice entre corchetes puede ser una variable.

In [5]:
i = 1
fruit[i]

O una expresión que contiene variables y operadores.

In [6]:
fruit[i+1]

Pero el valor del índice tiene que ser un entero -- de lo contrario obtienes un `TypeError`.

In [7]:
%%expect TypeError

fruit[1.5]

Como vimos en el capítulo 1, podemos usar la función integrada `len` para obtener la longitud de una cadena.

In [8]:
n = len(fruit)
n

Para obtener la última letra de una cadena, podrías sentir la tentación de escribir esto:

In [9]:
%%expect IndexError

fruit[n]

Pero eso causa un `IndexError` porque no hay ninguna letra en `'banana'` con el índice 6. Como empezamos a contar en `0`, las seis letras están numeradas de `0` a `5`. Para obtener el último carácter, tienes que restar `1` a `n`:

In [10]:
fruit[n-1]

Pero hay una forma más sencilla.
Para obtener la última letra de una cadena, puedes usar un índice negativo, que cuenta hacia atrás desde el final.

In [11]:
fruit[-1]

El índice `-1` selecciona la última letra, `-2` selecciona la penúltima, y así sucesivamente.

## Segmentos de cadenas

Un segmento de una cadena se llama **segmento**.
Seleccionar un segmento es similar a seleccionar un carácter.

In [12]:
fruit = 'banana'
fruit[0:3]

El operador `[n:m]` devuelve la parte de la cadena desde el carácter `n`-ésimo
hasta el carácter `m`-ésimo, incluyendo el primero pero excluyendo el segundo.
Este comportamiento es contraintuitivo, pero puede ayudar imaginar que los índices apuntan *entre* los caracteres, como en esta figura:

In [13]:
from diagram import make_binding, Element, Value

binding = make_binding("fruit", ' b a n a n a ')
elements = [Element(Value(i), None) for i in range(7)]

In [14]:
import matplotlib.pyplot as plt
from diagram import diagram, adjust
from matplotlib.transforms import Bbox

width, height, x, y = [1.35, 0.54, 0.23, 0.39]

ax = diagram(width, height)
bbox = binding.draw(ax, x, y)
bboxes = [bbox]

def draw_elts(x, y, elements):
    for elt in elements:
        bbox = elt.draw(ax, x, y, draw_value=False)
        bboxes.append(bbox)

        x1 = (bbox.xmin + bbox.xmax) / 2
        y1 = bbox.ymax + 0.02
        y2 = y1 + 0.14
        handle = plt.plot([x1, x1], [y1, y2], ':', lw=0.5, color='gray')
        x += 0.105
    
draw_elts(x + 0.48, y - 0.25, elements)
bbox = Bbox.union(bboxes)
# adjust(x, y, bbox)

Por ejemplo, el segmento `[3:6]` selecciona las letras `ana`, lo que significa que `6` es legal como parte de un segmento, pero no como índice.


Si omites el primer índice, el segmento empieza al principio de la cadena.

In [15]:
fruit[:3]

Si omites el segundo índice, el segmento va hasta el final de la cadena:

In [16]:
fruit[3:]

Si el primer índice es mayor o igual que el segundo, el resultado es una **cadena vacía**, representado por dos comillas:

In [17]:
fruit[3:3]

Una cadena vacío no contiene caracteres y tiene longitud 0.

Continuando con este ejemplo, ¿qué crees que significa `fruit[:]`? Pruébalo y
mira.

In [18]:
fruit[:]

## Las cadenas son inmutables

Es tentador usar el operador `[]` en el lado izquierdo de una
asignación, con la intención de cambiar un carácter en una cadena, así:

In [19]:
%%expect TypeError

greeting = 'Hello, world!'
greeting[0] = 'J'

El resultado es un `TypeError`.
En el mensaje de error, el "objeto" es la cadena y el "item" es el carácter
que intentamos asignar.
Por ahora, un **objeto** es lo mismo que un valor, pero refinaremos esa definición más adelante.

La razón de este error es que las cadenas son **inmutables**, lo que significa que no puedes cambiar una cadena existente.
Lo máximo que puedes hacer es crear una cadena nueva que sea una variación del original.

In [20]:
new_greeting = 'J' + greeting[1:]
new_greeting

Este ejemplo concatena una nueva primera letra con un segmento de `greeting`.
No tiene ningún efecto sobre la cadena original.

In [21]:
greeting

## Comparación de cadenas

Los operadores relacionales funcionan con cadenas. Para ver si dos cadenas son
iguales, podemos usar el operador `==`.

In [22]:
word = 'banana'

if word == 'banana':
    print('All right, banana.')

Otras operaciones relacionales son útiles para poner palabras en orden alfabético:

In [23]:
def compare_word(word):
    if word < 'banana':
        print(word, 'comes before banana.')
    elif word > 'banana':
        print(word, 'comes after banana.')
    else:
        print('All right, banana.')

In [24]:
compare_word('apple')

Python no trata las letras mayúsculas y minúsculas de la misma manera que
lo hacen las personas. Todas las letras mayúsculas van antes que todas las
minúsculas, así que:

In [25]:
compare_word('Pineapple')

Para resolver este problema, podemos convertir las cadenas a un formato estándar, como todo en minúsculas, antes de realizar la comparación.
Tenlo en cuenta si tienes que defenderte de un hombre armado con una piña.

## Métodos de cadenas

Las cadenas proporcionan métodos que realizan una variedad de operaciones útiles.
Un método es similar a una función -- recibe argumentos y devuelve un valor -- pero la sintaxis es diferente.
Por ejemplo, el método `upper` recibe una cadena y devuelve una cadena nueva con todas las letras en mayúsculas.

En lugar de la sintaxis de función `upper(word)`, usa la sintaxis de método `word.upper()`.

In [26]:
word = 'banana'
new_word = word.upper()
new_word

Este uso del operador punto especifica el nombre del método, `upper`, y el nombre de la cadena al que aplicar el método, `word`.
Los paréntesis vacíos indican que este método no toma argumentos.

Una llamada a método se llama **invocación**; en este caso, diríamos que estamos invocando `upper` sobre `word`.

## Escribir archivos

Los operadores y métodos de cadena son útiles para leer y escribir archivos de texto.
Como ejemplo, trabajaremos con el texto de *Dracula*, una novela de Bram Stoker que está disponible en Project Gutenberg (<https://www.gutenberg.org/ebooks/345>).

In [27]:
import os

if not os.path.exists('pg345.txt'):
    !wget https://www.gutenberg.org/cache/epub/345/pg345.txt

He descargado el libro en un archivo de texto plano llamado `pg345.txt`, que podemos abrir para lectura así:

In [28]:
reader = open('pg345.txt')

Además del texto del libro, este archivo contiene una sección al principio con información sobre el libro y una sección al final con información sobre la licencia.
Antes de procesar el texto, podemos eliminar este material extra encontrando las líneas especiales al principio y al final que empiezan con `'***'`.

La siguiente función recibe una línea y comprueba si es una de las líneas especiales.
Usa el método `startswith`, que comprueba si una cadena empieza con una secuencia determinada de caracteres.

In [29]:
def is_special_line(line):
    return line.startswith('*** ')

Podemos usar esta función para recorrer las líneas del archivo e imprimir solo las líneas especiales.

In [30]:
for line in reader:
    if is_special_line(line):
        print(line.strip())

Ahora creemos un archivo nuevo, llamado `pg345_cleaned.txt`, que contenga solo el texto del libro.
Para recorrer el libro de nuevo, tenemos que abrirlo otra vez para lectura.
Y, para escribir un archivo nuevo, podemos abrirlo para escritura.

In [31]:
reader = open('pg345.txt')
writer = open('pg345_cleaned.txt', 'w')

`open` recibe un parámetro opcional que especifica el "modo" -- en este ejemplo, `'w'` indica que estamos abriendo el archivo para escritura.
Si el archivo no existe, se creará; si ya existe, el contenido será reemplazado.

Como primer paso, recorreremos el archivo hasta encontrar la primera línea especial.

In [32]:
for line in reader:
    if is_special_line(line):
        break

La sentencia `break` "rompe" el bucle -- es decir, hace que el bucle termine inmediatamente, antes de llegar al final del archivo.

Cuando el bucle termina, `line` contiene la línea especial que hizo que la condición fuera verdadera.

In [33]:
line

Como `reader` lleva la cuenta de en qué parte del archivo está, podemos usar un segundo bucle para continuar donde lo dejamos.

El siguiente bucle lee el resto del archivo, una línea cada vez.
Cuando encuentra la línea especial que indica el final del texto, rompe el bucle.
En caso contrario, escribe la línea en el archivo de salida.

In [34]:
for line in reader:
    if is_special_line(line):
        break
    writer.write(line)

Cuando este bucle termina, `line` contiene la segunda línea especial.

In [35]:
line

En este punto `reader` y `writer` siguen abiertos, lo que significa que podríamos seguir leyendo líneas de `reader` o escribiendo líneas en `writer`.
Para indicar que hemos terminado, podemos cerrar ambos archivos invocando el método `close`.

In [36]:
reader.close()
writer.close()

Para comprobar si este proceso tuvo éxito, podemos leer las primeras líneas del archivo nuevo que acabamos de crear.

In [37]:
for line in open('pg345_cleaned.txt'):
    line = line.strip()
    if len(line) > 0:
        print(line)
    if line.endswith('Stoker'):
        break

El método `endswith` comprueba si una cadena termina con una secuencia determinada de caracteres.

## Buscar y reemplazar

En la traducción islandesa de *Dracula* de 1901, el nombre de uno de los personajes se cambió de "Jonathan" a "Thomas".
Para hacer este cambio en la versión inglesa, podemos recorrer el libro, usar el método `replace` para reemplazar un nombre por otro y escribir el resultado en un archivo nuevo.

Empezaremos contando las líneas en la versión limpia del archivo.

In [38]:
total = 0
for line in open('pg345_cleaned.txt'):
    total += 1
    
total

Para ver si una línea contiene "Jonathan", podemos usar el operador `in`, que comprueba si esta secuencia de caracteres aparece en cualquier parte de la línea.

In [39]:
total = 0
for line in open('pg345_cleaned.txt'):
    if 'Jonathan' in line:
        total += 1

total

Hay 199 líneas que contienen el nombre, pero ese no es exactamente el número total de veces que aparece, porque puede aparecer más de una vez en una línea.
Para obtener el total, podemos usar el método `count`, que devuelve el número de veces que una secuencia aparece en una cadena.

In [40]:
total = 0
for line in open('pg345_cleaned.txt'):
    total += line.count('Jonathan')

total

Ahora podemos reemplazar `'Jonathan'` por `'Thomas'` así:

In [41]:
writer = open('pg345_replaced.txt', 'w')

for line in open('pg345_cleaned.txt'):
    line = line.replace('Jonathan', 'Thomas')
    writer.write(line)

El resultado es un archivo nuevo llamado `pg345_replaced.txt` que contiene una versión de *Dracula* donde Jonathan Harker se llama Thomas.

In [42]:
total = 0
for line in open('pg345_replaced.txt'):
    total += line.count('Thomas')

total

## Expresiones regulares

Si sabemos exactamente qué secuencia de caracteres estamos buscando, podemos usar el operador `in` para encontrarla y el método `replace` para reemplazarla.
Pero hay otra herramienta, llamada **expresión regular**, que también puede realizar estas operaciones -- y muchas más.

Para demostrarlo, empezaré con un ejemplo sencillo y avanzaremos poco a poco.
Supongamos, otra vez, que queremos encontrar todas las líneas que contienen una palabra concreta.
Para variar, busquemos referencias al personaje titular del libro, el conde Dracula.
Aquí tienes una línea que lo menciona.

In [43]:
text = "I am Dracula; and I bid you welcome, Mr. Harker, to my house."

Y aquí está el **patrón** que usaremos para buscar.

In [44]:
pattern = 'Dracula'

Un módulo llamado `re` proporciona funciones relacionadas con expresiones regulares.
Podemos importarlo así y usar la función `search` para comprobar si el patrón aparece en el texto.

In [45]:
import re

result = re.search(pattern, text)
result

Si el patrón aparece en el texto, `search` devuelve un objeto `Match` que contiene los resultados de la búsqueda.
Entre otra información, tiene una variable llamada `string` que contiene el texto en el que se buscó.

In [46]:
result.string

También proporciona un método llamado `group` que devuelve la parte del texto que coincidió con el patrón.

In [47]:
result.group()

Y proporciona un método llamado `span` que devuelve el índice en el texto donde empieza y termina el patrón.

In [48]:
result.span()

Si el patrón no aparece en el texto, el valor de retorno de `search` es `None`.

In [49]:
result = re.search('Count', text)
print(result)

Así que podemos comprobar si la búsqueda tuvo éxito comprobando si el resultado es `None`.

In [50]:
result == None

Juntando todo eso, aquí tienes una función que recorre las líneas del libro hasta encontrar una que coincida con el patrón dado, y devuelve el objeto `Match`.

In [51]:
def find_first(pattern):
    for line in open('pg345_cleaned.txt'):
        result = re.search(pattern, line)
        if result != None:
            return result

Podemos usarla para encontrar la primera mención de un personaje.

In [52]:
result = find_first('Harker')
result.string

Para este ejemplo, no tuvimos que usar expresiones regulares -- podríamos haber hecho lo mismo más fácilmente con el operador `in`.
Pero las expresiones regulares pueden hacer cosas que el operador `in` no puede.

Por ejemplo, si el patrón incluye el carácter de barra vertical, `'|'`, puede coincidir con la secuencia de la izquierda o con la secuencia de la derecha.
Supongamos que queremos encontrar la primera mención de Mina Murray en el libro, pero no estamos seguros de si se la menciona por su nombre o por su apellido.
Podemos usar el siguiente patrón, que coincide con cualquiera de los dos nombres.

In [53]:
pattern = 'Mina|Murray'
result = find_first(pattern)
result.string

Podemos usar un patrón como este para ver cuántas veces se menciona a un personaje por cualquiera de los dos nombres.
Aquí tienes una función que recorre el libro y cuenta el número de líneas que coinciden con el patrón dado.

In [54]:
def count_matches(pattern):
    count = 0
    for line in open('pg345_cleaned.txt'):
        result = re.search(pattern, line)
        if result != None:
            count += 1
    return count

Ahora veamos cuántas veces se menciona a Mina.

In [55]:
count_matches('Mina|Murray')

El carácter especial `'^'` coincide con el principio de una cadena, así que podemos encontrar una línea que empieza con un patrón dado.

In [56]:
result = find_first('^Dracula')
result.string

Y el carácter especial `'$'` coincide con el final de una cadena, así que podemos encontrar una línea que termina con un patrón dado (ignorando el salto de línea del final).

In [57]:
result = find_first('Harker$')
result.string

## Sustitución de cadenas

Bram Stoker nació en Irlanda, y cuando *Dracula* se publicó en 1897, vivía en Inglaterra.
Así que esperaríamos que usara la ortografía británica de palabras como "centre" y "colour".
Para comprobarlo, podemos usar el siguiente patrón, que coincide con "centre" o con la ortografía estadounidense "center".

In [58]:
pattern = 'cent(er|re)'

En este patrón, los paréntesis encierran la parte del patrón a la que se aplica la barra vertical.
Así que este patrón coincide con una secuencia que empieza con `'cent'` y termina con `'er'` o con `'re'`.

In [59]:
result = find_first(pattern)
result.string

Como esperábamos, usó la ortografía británica.

También podemos comprobar si usó la ortografía británica de "colour".
El siguiente patrón usa el carácter especial `'?'`, que significa que el carácter anterior es opcional.

In [60]:
pattern = 'colou?r'

Este patrón coincide con "colour" con la `'u'` o con "color" sin ella.

In [61]:
result = find_first(pattern)
line = result.string
line

De nuevo, como esperábamos, usó la ortografía británica.

Ahora supongamos que queremos producir una edición del libro con ortografía estadounidense.
Podemos usar la función `sub` del módulo `re`, que hace **sustitución de cadenas**.

In [62]:
re.sub(pattern, 'color', line)

El primer argumento es el patrón que queremos encontrar y reemplazar, el segundo es aquello con lo que queremos reemplazarlo, y el tercero es la cadena en el que queremos buscar.
En el resultado, puedes ver que "colour" ha sido reemplazado por "color".

In [63]:
# I used this function to search for lines to use as examples

def all_matches(pattern):
    for line in open('pg345_cleaned.txt'):
        result = re.search(pattern, line)
        if result:
            print(line.strip())

In [64]:
# Here's the pattern I used (which uses some features we haven't seen)

names = r'(?<!\.\s)[A-Z][a-zA-Z]+'

all_matches(names)

## Depuración

Cuando lees y escribes archivos, depurar puede ser complicado.
Si trabajas en un Jupyter notebook, puedes usar **comandos de shell** para ayudar.
Por ejemplo, para mostrar las primeras líneas de un archivo, puedes usar el comando `!head`, así:

In [65]:
!head pg345_cleaned.txt

El signo de exclamación inicial, `!`, indica que esto es un comando de shell, que no forma parte de Python.
Para mostrar las últimas líneas, puedes usar `!tail`.

In [66]:
!tail pg345_cleaned.txt

Cuando trabajas con archivos grandes, depurar puede ser difícil porque puede haber demasiada salida para revisarla a mano.
Una buena estrategia de depuración es empezar con solo una parte del archivo, hacer que el programa funcione, y luego ejecutarlo con el archivo completo.

Para crear un archivo pequeño que contenga parte de un archivo más grande, podemos usar `!head` de nuevo con el operador de redirección, `>`, que indica que los resultados deben escribirse en un archivo en lugar de mostrarse.

In [67]:
!head pg345_cleaned.txt > pg345_cleaned_10_lines.txt

Por defecto, `!head` lee las primeras 10 líneas, pero recibe un argumento opcional que indica el número de líneas a leer.

In [68]:
!head -100 pg345_cleaned.txt > pg345_cleaned_100_lines.txt

Este comando de shell lee las primeras 100 líneas de `pg345_cleaned.txt` y las escribe en un archivo llamado `pg345_cleaned_100_lines.txt`.

Nota: Los comandos de shell `!head` y `!tail` no están disponibles en todos los sistemas operativos.
Si no te funcionan, podemos escribir funciones similares en Python.
Consulta el primer ejercicio al final de este capítulo para ver sugerencias.

## Glosario

**secuencia:**
 Una colección ordenada de valores donde cada valor se identifica mediante un índice entero.

**carácter:**
Un elemento de una cadena, incluidas letras, números y símbolos.

**índice:**
 Un valor entero usado para seleccionar un elemento en una secuencia, como un carácter en una cadena. En Python los índices empiezan desde `0`.

**segmento:**
 Una parte de una cadena especificada por un rango de índices.

**cadena vacía:**
Una cadena que no contiene caracteres y tiene longitud `0`.

**objeto:**
 Algo a lo que una variable puede referirse. Un objeto tiene un tipo y un valor.

**inmutable:**
Si los elementos de un objeto no se pueden cambiar, el objeto es inmutable.

**invocación:**
 Una expresión -- o parte de una expresión -- que llama a un método.

**expresión regular:**
Una secuencia de caracteres que define un patrón de búsqueda.

**patrón:**
Una regla que especifica los requisitos que una cadena debe cumplir para constituir una coincidencia.

**sustitución de cadenas:**
Reemplazo de una cadena, o parte de una cadena, por otra cadena.

**comando de shell:**
Una sentencia en un lenguaje de shell, que es un lenguaje usado para interactuar con un sistema operativo.

## Ejercicios

In [69]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

In [70]:
download('https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt');

### Pregunta a un asistente virtual

En este capítulo, apenas hemos arañado la superficie de lo que pueden hacer las expresiones regulares.
Para hacerte una idea de lo que es posible, pregunta a un asistente virtual: "¿Cuáles son los caracteres especiales más comunes que se usan en las expresiones regulares de Python?"

También puedes pedir un patrón que coincida con tipos concretos de cadenas.
Por ejemplo, prueba a preguntar:

* Escribe una expresión regular de Python que coincida con un número de teléfono de 10 dígitos con guiones.

* Escribe una expresión regular de Python que coincida con una dirección con un número y un nombre de calle, seguida de `ST` o `AVE`.

* Escribe una expresión regular de Python que coincida con un nombre completo con cualquier título común como `Mr` o `Mrs`, seguido de cualquier número de nombres que empiecen con mayúsculas, posiblemente con guiones entre algunos nombres.

Y si quieres ver algo más complicado, prueba a pedir una expresión regular que coincida con cualquier URL legal.

Una expresión regular a menudo tiene la letra `r` antes de la comilla, lo que indica que es una "cadena sin procesar".
Para más información, pregunta a un asistente virtual: "¿Qué es una cadena sin procesar en Python?"

In [71]:
from doctest import run_docstring_examples

def run_doctests(func):
    run_docstring_examples(func, globals(), name=func.__name__)

### Ejercicio

Mira si puedes escribir una función que haga lo mismo que el comando de shell `!head`.
Debe recibir como argumentos el nombre de un archivo que leer, el número de líneas que leer y el nombre del archivo donde escribir las líneas.
Si el tercer parámetro es `None`, debe mostrar las líneas en lugar de escribirlas en un archivo.

Considera pedir ayuda a un asistente virtual, pero si lo haces, dile que no use una sentencia `with` ni una sentencia `try`.

In [72]:
# Solution goes here

Puedes usar los siguientes ejemplos para probar tu función.

In [73]:
head('pg345_cleaned.txt', 10)

In [74]:
head('pg345_cleaned.txt', 100, 'pg345_cleaned_100_lines.txt')

In [75]:
!tail pg345_cleaned_100_lines.txt

### Ejercicio

"Wordle" es un juego de palabras online donde el objetivo es adivinar una palabra de cinco letras en seis intentos o menos.
Cada intento tiene que ser reconocido como una palabra, sin incluir nombres propios.
Después de cada intento, obtienes información sobre cuáles de las letras que adivinaste aparecen en la palabra objetivo, y cuáles están en la posición correcta.

Por ejemplo, supongamos que la palabra objetivo es `MOWER` y que intentas `TRIED`.
Aprenderías que `E` está en la palabra y en la posición correcta, `R` está en la palabra pero no en la posición correcta, y `T`, `I` y `D` no están en la palabra.

Como ejemplo distinto, supongamos que has intentado las palabras `SPADE` y `CLERK`, y has aprendido que `E` está en la palabra, pero no en ninguna de esas posiciones, y que ninguna de las otras letras aparece en la palabra.
De las palabras de la lista, ¿cuántas podrían ser la palabra objetivo?
Escribe una función llamada `check_word` que reciba una palabra de cinco letras y compruebe si podría ser la palabra objetivo, dadas estas conjeturas.

In [76]:
# Solution goes here

Puedes usar cualquiera de las funciones del capítulo anterior, como `uses_any`.

In [77]:
def uses_any(word, letters):
    for letter in word.lower():
        if letter in letters.lower():
            return True
    return False

Puedes usar el siguiente bucle para probar tu función.

In [78]:
for line in open('words.txt'):
    word = line.strip()
    if len(word) == 5 and check_word(word):
        print(word)

### Ejercicio

Continuando con el ejercicio anterior, supongamos que intentas la palabra `TOTEM` y aprendes que la `E` *todavía* no está en el lugar correcto, pero la `M` sí. ¿Cuántas palabras quedan?

In [79]:
# Solution goes here

In [80]:
# Solution goes here

### Ejercicio

*The Count of Monte Cristo* es una novela de Alexandre Dumas que se considera un clásico.
Sin embargo, en la introducción de una traducción inglesa del libro, el escritor Umberto Eco confiesa que le pareció "una de las novelas peor escritas de todos los tiempos".

En particular, dice que es "desvergonzada en su repetición del mismo adjetivo", y menciona en particular el número de veces que "sus personajes se estremecen o palidecen".

Para ver si su objeción es válida, contemos el número de líneas que contienen la palabra `pale` en cualquier forma, incluidas `pale`, `pales`, `paled` y `paleness`, así como la palabra relacionada `pallor`.
Usa una sola expresión regular que coincida con cualquiera de estas palabras.
Como desafío adicional, asegúrate de que no coincida con otras palabras, como `impale` -- quizá quieras pedir ayuda a un asistente virtual.

La siguiente celda descarga el libro de Project Gutenberg <https://www.gutenberg.org/ebooks/1184>.

In [81]:
import os

if not os.path.exists('pg1184.txt'):
    !wget https://www.gutenberg.org/cache/epub/1184/pg1184.txt

La siguiente celda ejecuta una función que lee el archivo de Project Gutenberg y escribe un archivo que contiene solo el texto del libro, no la información añadida sobre el libro.

In [82]:
def clean_file(input_file, output_file):
    reader = open(input_file)
    writer = open(output_file, 'w')

    for line in reader:
        if is_special_line(line):
            break

    for line in reader:
        if is_special_line(line):
            break
        writer.write(line)
        
    reader.close()
    writer.close()

clean_file('pg1184.txt', 'pg1184_cleaned.txt')

In [83]:
# Solution goes here

In [84]:
# Solution goes here

In [85]:
# Solution goes here

Según este recuento, estas palabras aparecen en `223` líneas del libro, así que el señor Eco quizá tenga razón.

[Think Python: 3rd Edition](https://allendowney.github.io/ThinkPython/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

Traducción al español por midudev (Miguel Ángel Durán).

Licencia del código: [MIT License](https://mit-license.org/)

Licencia del texto: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)